# Variant D — CoT-Augmented Hybrid (D-Human / D-CoT / D-Blend)

**Architecture:** identical to Variant C — DeBERTa-v3-base with two heads:
1. Classification head: NLI label prediction from `[CLS]`
2. MLM head: masked-token prediction on the **rationale segment** only

**Joint loss:** `0.7 × CE_label + 0.3 × MLM_rationale`  (α = 0.7, MLM prob = 0.15)

**Three sub-configs differ only in which text fills the rationale segment:**
- `D-Human` — `Explanation_1` (e-SNLI human explanation)
- `D-CoT`   — `cot_rationale_esnli_style` (LLM-generated CoT trace)
- `D-Blend` — per-row deterministic mix of the two, governed by `blend_ratio`

**Training data:** the CoT-complete rows from the merged subset — **765 of the 2,000-row
target have completed traces** (`Datasets/CoT/cot_traces.csv`), split 90/10 → **~689 train / ~76 dev**.
All other hyperparameters match A/B/C; only `warmup_steps` is recomputed (**4** instead of 3090)
for the smaller training set (689 ÷ 32 × 3 epochs = 63 total steps; 6% ≈ 4).

**Run order:** set `SUB_CONFIG` in Cell 5 to `"human"`, run cells 5–10, then re-run with `"cot"`, then `"blend"`. (Or use Cell 11 to run all three back-to-back in one session.)

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets accelerate torch

In [ ]:
# Cell 2-prep: Merge cot_traces_part1.csv + cot_traces_part2.csv into cot_traces.csv
# Run this cell ONCE after uploading both part files to Drive.
# Safe to re-run — it overwrites the merged file in place.
import os
import pandas as pd

# Adjust these paths if your part files live elsewhere on Drive.
PART1 = os.path.join(PROJECT_ROOT, "Datasets", "CoT", "cot_traces_part1.csv")
PART2 = os.path.join(PROJECT_ROOT, "Datasets", "CoT", "cot_traces_part2.csv")
MERGED = os.path.join(PROJECT_ROOT, "Datasets", "CoT", "cot_traces.csv")

os.makedirs(os.path.dirname(MERGED), exist_ok=True)

df1 = pd.read_csv(PART1)
df2 = pd.read_csv(PART2)
combined = pd.concat([df1, df2], ignore_index=True)

# Keep the version WITH a CoT trace when pair_id appears in both files.
cot_col = "cot_rationale_esnli_style"
combined["_has_cot"] = combined[cot_col].notna() & (combined[cot_col].str.strip() != "")
combined = combined.sort_values("_has_cot", ascending=False).drop_duplicates(subset="pair_id", keep="first")
combined = combined.drop(columns=["_has_cot"]).sort_values("pair_id").reset_index(drop=True)

combined.to_csv(MERGED, index=False)

total = len(combined)
cot_complete = (combined[cot_col].notna() & (combined[cot_col].str.strip() != "")).sum()
print(f"Merged CSV written to: {MERGED}")
print(f"Total unique rows:     {total:,}")
print(f"Rows with CoT traces:  {cot_complete:,}  (these are the training candidates)")
print(f"Rows without CoT:      {total - cot_complete:,}  (will be filtered out by load_cot_subset)")

In [ ]:
# Cell 2: Mount Google Drive and set project path
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

# Set this to where your project folder lives on Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

In [ ]:
# Cell 3: Verify GPU
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:     ", torch.cuda.get_device_name(0))
    print("GPU memory:   ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU detected.")
    print("Go to Runtime > Change runtime type > GPU > Save and re-run.")

In [ ]:
# Cell 4: Load CoT subset and stratified-split into train/dev
from training.config_d import VariantDConfig
from data.preprocess_d import load_cot_subset, split_cot_subset

# A throwaway config for the *loading* step — sub_config doesn't affect what's loaded.
load_cfg = VariantDConfig()

print("Loading CoT subset...")
full_df = load_cot_subset(load_cfg)
assert len(full_df) > 0, "CoT subset is empty after filtering."
print(f"  Total rows after filter: {len(full_df)}")

print("Per-label counts:")
print(full_df["gold_label"].value_counts().sort_index())

train_df, dev_df = split_cot_subset(full_df, dev_frac=load_cfg.dev_frac, seed=load_cfg.split_seed)
print(f"\nTrain rows: {len(train_df):,}")
print(f"Dev rows:   {len(dev_df):,}")

print("\nTrain label balance:")
print(train_df["gold_label"].value_counts().sort_index())
print("\nDev label balance:")
print(dev_df["gold_label"].value_counts().sort_index())

print("\nSample row:")
row = train_df.iloc[0]
print(f"  pair_id:     {row['pair_id']}")
print(f"  Premise:     {row['Sentence1']}")
print(f"  Hypothesis:  {row['Sentence2']}")
print(f"  Label:       {row['gold_label']}")
print(f"  Human expl:  {row['Explanation_1']}")
print(f"  CoT trace:   {row['cot_rationale_esnli_style']}")

In [ ]:
# Cell 5: Pick the sub-config to train, then build tokenizer / dataset / model
# Re-run this cell (and all subsequent cells) with each value: "human", "cot", "blend".
from models.common import load_tokenizer
from models.variant_d import DeBERTaForVariantD
from data.preprocess_d import ESNLIVariantDDataset

SUB_CONFIG = "human"     # one of {"human", "cot", "blend"}
BLEND_RATIO = 0.5        # only used when SUB_CONFIG == "blend"

config = VariantDConfig(sub_config=SUB_CONFIG, blend_ratio=BLEND_RATIO)

print("Loading tokenizer...")
tokenizer = load_tokenizer(config.model_name)
print(f"  is_fast: {tokenizer.is_fast}")

print("\nBuilding datasets...")
train_dataset = ESNLIVariantDDataset(train_df, tokenizer, config, is_train=True)
eval_dataset  = ESNLIVariantDDataset(dev_df,   tokenizer, config, is_train=False)
print(f"  Train dataset: {len(train_dataset):,} examples")
print(f"  Eval dataset:  {len(eval_dataset):,} examples")

# Sanity check: confirm rationale source is what we asked for, and masking fires
sample = train_dataset[0]
masked = (sample['mlm_labels'] != -100).sum().item()
chosen = train_dataset._select_rationale(0)
print(f"\nSanity check on one sample (sub_config={SUB_CONFIG}):")
print(f"  input_ids shape:    {sample['input_ids'].shape}")
print(f"  label:              {sample['labels'].item()} ({config.id2label[sample['labels'].item()]})")
print(f"  rationale tokens masked: {masked}  (must be > 0)")
print(f"  rationale text used: {chosen[:120]}{'...' if len(chosen) > 120 else ''}")

print(f"\nLoading model: {config.model_name}")
model = DeBERTaForVariantD(
    model_name=config.model_name,
    num_labels=config.num_labels,
    alpha=config.alpha,
)
total = sum(p.numel() for p in model.parameters())
print(f"  Total parameters:   {total:,}")
print(f"  Alpha (cls weight): {config.alpha}  |  MLM weight: {1 - config.alpha}")

In [ ]:
# Cell 6: Check for existing checkpoint (per sub-config; safe across Colab disconnects)
import glob
import shutil

checkpoint_dir = str(config.get_path("checkpoint_dir"))
os.makedirs(checkpoint_dir, exist_ok=True)

resume_from = None
existing = sorted(glob.glob(os.path.join(checkpoint_dir, "checkpoint-*")))

for ckpt in existing:
    model_pt = os.path.join(ckpt, "model.pt")
    if os.path.isfile(model_pt):
        resume_from = ckpt
    else:
        print(f"Removing invalid checkpoint (no model.pt): {ckpt}")
        shutil.rmtree(ckpt)

if resume_from:
    print(f"Found valid checkpoint: {resume_from}")
    print("Training will resume from this point.")
else:
    print("No valid checkpoint found — training from scratch.")

print(f"\nCheckpoint directory: {checkpoint_dir}")

In [ ]:
# Cell 7: Train
from transformers import TrainingArguments
from training.train import MultiTaskTrainer, compute_metrics

# Fix 1: prevent GPU memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Fix 2: release any leftover GPU memory from earlier cells
torch.cuda.empty_cache()

# Fix 3: ensure all parameters are float32 so the fp16 gradient scaler works correctly
model = model.float()

steps_per_epoch = len(train_dataset) // (config.per_device_train_batch_size * config.gradient_accumulation_steps)
total_steps     = steps_per_epoch * config.num_train_epochs

print("Training config:")
print(f"  Sub-config:               {config.sub_config}")
print(f"  Batch size (per device):  {config.per_device_train_batch_size}")
print(f"  Gradient accumulation:    {config.gradient_accumulation_steps}")
print(f"  Effective batch size:     {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
print(f"  Steps per epoch:          {steps_per_epoch}")
print(f"  Total optimizer steps:    {total_steps}")
print(f"  Warmup steps:             {config.warmup_steps}")
print()
print("Variant D trains on ~689 rows -> each sub-config takes ~3-5 min on a T4.")
print()

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_steps=config.warmup_steps,
    fp16=config.fp16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=config.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_num_workers=0,
    logging_steps=25,
    report_to="none",
    remove_unused_columns=False,
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)
# Note: not setting trainer._variant_c_config — that would write a misleadingly-named
# variant_c_config.json next to each checkpoint. Cell 8 writes variant_d_config.json explicitly.

trainer.train(resume_from_checkpoint=resume_from)
print("\nTraining complete!")

In [ ]:
# Cell 8: Save final model and config to Drive (sub-config-scoped folder)
import json
from dataclasses import asdict

final_dir = str(config.get_path("output_dir"))
os.makedirs(final_dir, exist_ok=True)

model_path = os.path.join(final_dir, "model.pt")
torch.save(model.state_dict(), model_path)
print(f"Model saved:  {model_path}")

config_path = os.path.join(final_dir, "variant_d_config.json")
with open(config_path, "w") as f:
    json.dump(asdict(config), f, indent=2, default=str)
print(f"Config saved: {config_path}")

In [ ]:
# Cell 9: Evaluate on e-SNLI test (premise + hypothesis only — no rationale at inference)
from evaluation.evaluate import evaluate_dataset
from data.preprocess import NLIDataset
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Loading e-SNLI test set...")
test_csv = config.get_path("esnli_test")
test_df_raw = pd.read_csv(test_csv, usecols=["gold_label", "Sentence1", "Sentence2", "Explanation_1"])
test_df_raw = test_df_raw[test_df_raw["gold_label"].isin({"entailment", "neutral", "contradiction"})].dropna(subset=["Sentence1", "Sentence2"]).reset_index(drop=True)

test_dataset = NLIDataset(
    premises=test_df_raw["Sentence1"].tolist(),
    hypotheses=test_df_raw["Sentence2"].tolist(),
    labels=[config.label2id[lbl] for lbl in test_df_raw["gold_label"].tolist()],
    tokenizer=tokenizer,
    max_length=config.max_seq_length,
)

print(f"Evaluating on {len(test_dataset):,} examples...")
esnli_acc = evaluate_dataset(model, test_dataset, batch_size=config.per_device_eval_batch_size, device=device)
print(f"\ne-SNLI test accuracy: {esnli_acc:.4f}  ({esnli_acc * 100:.2f}%)")

In [ ]:
# Cell 10: Cross-domain evaluation (SNLI, MultiNLI, ANLI) — same benchmarks as A/B/C
from data.preprocess import load_snli_test, load_multinli, load_anli
import json

results = {"esnli_test": esnli_acc, "sub_config": config.sub_config, "blend_ratio": config.blend_ratio}

print("Evaluating SNLI test...")
snli = load_snli_test(tokenizer, config.max_seq_length)
results["snli_test"] = evaluate_dataset(model, snli, config.per_device_eval_batch_size, device)
print(f"  {results['snli_test']:.4f}")

print("Evaluating MultiNLI matched...")
mnli_m = load_multinli(tokenizer, split="validation_matched", max_length=config.max_seq_length)
results["multinli_matched"] = evaluate_dataset(model, mnli_m, config.per_device_eval_batch_size, device)
print(f"  {results['multinli_matched']:.4f}")

print("Evaluating MultiNLI mismatched...")
mnli_mm = load_multinli(tokenizer, split="validation_mismatched", max_length=config.max_seq_length)
results["multinli_mismatched"] = evaluate_dataset(model, mnli_mm, config.per_device_eval_batch_size, device)
print(f"  {results['multinli_mismatched']:.4f}")

for r in ("r1", "r2", "r3"):
    print(f"Evaluating ANLI {r}...")
    anli = load_anli(tokenizer, round_tag=r, max_length=config.max_seq_length)
    results[f"anli_{r}"] = evaluate_dataset(model, anli, config.per_device_eval_batch_size, device)
    print(f"  {results[f'anli_{r}']:.4f}")

print("\n" + "=" * 50)
print(f"Variant D — sub_config = {config.sub_config}")
print("=" * 50)
print(f"{'Benchmark':<25} {'Accuracy':>10}")
print("-" * 50)
for k, v in results.items():
    if isinstance(v, float):
        print(f"{k:<25} {v * 100:>9.2f}%")

results_path = os.path.join(final_dir, "eval_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to: {results_path}")

## Optional: run all three sub-configs back-to-back

If your Colab session has time budget for all three runs (≈30 min total on T4), use the cell below instead of re-running cells 5–10 manually three times. It calls `train_all_sub_configs` from `training/train_d.py`, which trains D-Human, D-CoT, and D-Blend in sequence and writes each to its own `results/variant_d_<sub_config>/` folder. Evaluation is **not** included in this convenience path — re-run cells 9–10 per sub-config afterwards (or load each saved model and evaluate).

In [ ]:
# Cell 11 (optional): Train all three sub-configs back-to-back
from training.train_d import train_all_sub_configs

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

train_all_sub_configs({"blend_ratio": 0.5})
print("\nAll three sub-configs trained. Re-run cells 5–10 per sub-config to evaluate.")